# ⚽ Premier League Season Analysis

**Author:** Kevin Gutekunst
**Course:** Python Programming for Business  
**Date:** 10.5.2026

---

## Introduction

Football clubs invest heavily in players and tactics, but which factors actually separate the top teams from the rest?

**Goal:** Use Premier League 2023/24 final standings data to answer:
- Which teams scored and conceded the most?
- Is there a relationship between goals scored and final league position?
- How do the top 6 clubs compare across key stats?

Data is collected via the API-Football API and analysed using `pandas`, `matplotlib`, and `seaborn`.

## Data Pipeline Overview

The diagram below shows the full data pipeline used in this notebook.

![Data Pipeline](pipeline_diagram.png)

> *Flow: API collection → raw save → cleaning → analysis → export.*

---
## Part 1 — Imports and Setup

In [3]:
import os
import json
import requests
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

API_KEY = "c42debee9d79e20d0a7b5d86b6fcad6a"

if API_KEY:
    print(f"Key loaded: {API_KEY[:4]}...")
else:
    print("WARNING: API_KEY not found.")


Key loaded: c42d...


In [1]:
import os
import json
import requests
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from dotenv import load_dotenv

# Load the API key from the .env file
load_dotenv()
API_KEY = os.environ.get("API_KEY_FILE.txt")

if API_KEY:
    print(f"Key loaded: {API_KEY[:4]}...")
else:
    print("WARNING: API_KEY not found in .env file.")

---
## Part 2 — Data Collection via API

We use the [API-Football](https://www.api-football.com/) API to fetch Premier League 2023/24 standings.
The free tier allows 100 requests/day, which is more than enough for this project.

League ID 39 = Premier League, season 2023 = the 2023/24 season.

In [ ]:
def fetch_standings(league_id, season, api_key):
    """Fetch league standings from the API-Football API.

    Args:
        league_id: Numeric league identifier (39 = Premier League)
        season: Season start year (e.g. 2023 for 2023/24)
        api_key: API-Football authentication key

    Returns:
        List of team standing dictionaries as returned by the API
    """
    url = "https://v3.football.api-sports.io/standings"
    headers = {"x-apisports-key": api_key}
    params = {"league": league_id, "season": season}

    response = requests.get(url, headers=headers, params=params)
    response.raise_for_status()

    data = response.json()
    standings = data["response"][0]["league"]["standings"][0]
    return standings


if API_KEY:
    raw_standings = fetch_standings(league_id=39, season=2023, api_key=API_KEY)
    print(f"Fetched data for {len(raw_standings)} teams")
    print()
    print("Sample entry:", raw_standings[0])
else:
    print("No API key found — loading from raw_standings.json instead.")
    print("Run the alternative cell (Cell 7) below, or add your API key to .env")
    raise SystemExit("Stopping here: no API key. Use the load-from-file cell (Cell 7).")


In [ ]:
# Save the raw API response so reviewers can run the notebook without an API key
with open("raw_standings.json", "w") as f:
    json.dump(raw_standings, f, indent=2)

print("Raw data saved to raw_standings.json")

In [ ]:
# --- ALTERNATIVE: load from the saved file (use this if you have no API key) ---
# Comment out Cell 5 above and run this cell instead.

import os

_fallback_path = "raw_standings.json"

if not os.path.exists(_fallback_path):
    raise FileNotFoundError(
        f"'{_fallback_path}' not found. You need either:\n"
        "  1. An API key in .env  (run Cell 5), OR\n"
        "  2. A previously saved raw_standings.json in the same folder."
    )

with open(_fallback_path, "r") as f:
    raw_standings = json.load(f)

print(f"Loaded saved data for {len(raw_standings)} teams")


---
## Part 3 — Cleaning and Transformation

The raw response is a nested list of dictionaries. We extract the relevant fields and build a flat DataFrame.
We also add two derived columns that will be useful for analysis.

In [ ]:
def parse_standings(raw):
    """Extract relevant fields from the raw API response into a flat DataFrame.

    Args:
        raw: List of team dictionaries returned by the API

    Returns:
        pandas DataFrame with one row per team
    """
    rows = []
    for team in raw:
        rows.append({
            "position":       team["rank"],
            "team":           team["team"]["name"],
            "played":         team["all"]["played"],
            "wins":           team["all"]["win"],
            "draws":          team["all"]["draw"],
            "losses":         team["all"]["lose"],
            "goals_scored":   team["all"]["goals"]["for"],
            "goals_conceded": team["all"]["goals"]["against"],
            "goal_diff":      team["goalsDiff"],
            "points":         team["points"],
        })
    return pd.DataFrame(rows)


df = parse_standings(raw_standings)
df.head()

In [ ]:
# Win rate: share of played matches that were won
df["win_rate"] = (df["wins"] / df["played"]).round(3)

# Classify each team based on their final position
def classify_team(position):
    if position <= 4:
        return "Champions League"
    elif position <= 6:
        return "Europa / Conference"
    elif position >= 18:
        return "Relegated"
    else:
        return "Mid-table"

df["category"] = df["position"].apply(classify_team)

print("Shape:", df.shape)
print()
df

In [ ]:
# Quick data quality check
print("Missing values per column:")
print(df.isna().sum())
print()
df.info()

---
## Part 4 — Analysis

### 4.1 Summary Statistics

In [ ]:
stats_cols = ["goals_scored", "goals_conceded", "goal_diff", "points", "win_rate"]
df[stats_cols].describe().round(2)

In [ ]:
# Average stats grouped by league position category
df.groupby("category")[["goals_scored", "goals_conceded", "points", "win_rate"]].mean().round(2)

### 4.2 Goals Scored vs Goals Conceded

In [ ]:
category_colors = {
    "Champions League":    "#2ecc71",
    "Europa / Conference": "#3498db",
    "Mid-table":           "#95a5a6",
    "Relegated":           "#e74c3c",
}

fig, ax = plt.subplots(figsize=(11, 7))

for cat, color in category_colors.items():
    subset = df[df["category"] == cat]
    ax.scatter(
        subset["goals_scored"], subset["goals_conceded"],
        color=color, label=cat, s=120, edgecolors="white", linewidths=0.8, zorder=3
    )
    for _, row in subset.iterrows():
        ax.annotate(
            row["team"], (row["goals_scored"], row["goals_conceded"]),
            textcoords="offset points", xytext=(6, 3), fontsize=7.5, color="#333333"
        )

# Reference diagonal: teams above the line concede more than they score
lim = max(df["goals_scored"].max(), df["goals_conceded"].max()) + 10
ax.plot([0, lim], [0, lim], "--", color="#bdc3c7", linewidth=1, label="Equal goals")

ax.set_xlabel("Goals Scored")
ax.set_ylabel("Goals Conceded")
ax.set_title("Premier League 2023/24 — Goals Scored vs Goals Conceded")
ax.legend(loc="upper left")

plt.tight_layout()
plt.savefig("plot_goals_scatter.png", dpi=150)
plt.show()

### 4.3 Final Points Tally — All Teams

In [ ]:
df_sorted = df.sort_values("position")
colors = [category_colors[cat] for cat in df_sorted["category"]]

fig, ax = plt.subplots(figsize=(14, 6))

bars = ax.barh(df_sorted["team"], df_sorted["points"], color=colors, edgecolor="white", height=0.7)

# Add point labels at the end of each bar
for bar, val in zip(bars, df_sorted["points"]):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height() / 2,
            str(val), va="center", fontsize=9)

ax.invert_yaxis()
ax.set_xlabel("Points")
ax.set_title("Premier League 2023/24 — Final Points Tally")

patches = [mpatches.Patch(color=c, label=l) for l, c in category_colors.items()]
ax.legend(handles=patches, loc="lower right")

plt.tight_layout()
plt.savefig("plot_points_bar.png", dpi=150)
plt.show()

### 4.4 Correlation — Goals Scored vs Points

In [ ]:
corr = df["goals_scored"].corr(df["points"]).round(3)
print(f"Pearson correlation (goals scored → points): {corr}")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))

sns.regplot(
    data=df, x="goals_scored", y="points",
    scatter_kws={"s": 80, "color": "#3498db", "edgecolors": "white"},
    line_kws={"color": "#e74c3c", "linewidth": 2},
    ax=ax
)

for _, row in df.iterrows():
    ax.annotate(row["team"], (row["goals_scored"], row["points"]),
                textcoords="offset points", xytext=(5, 3), fontsize=7.5, color="#555")

ax.set_xlabel("Goals Scored")
ax.set_ylabel("Points")
ax.set_title(f"Goals Scored vs Points  (r = {corr})")

plt.tight_layout()
plt.savefig("plot_correlation.png", dpi=150)
plt.show()

### 4.5 Top 6 — Wins, Draws, Losses

In [ ]:
top6 = df[df["position"] <= 6].sort_values("position")

metrics = ["wins", "draws", "losses"]
stack_colors = ["#2ecc71", "#f39c12", "#e74c3c"]

fig, ax = plt.subplots(figsize=(11, 5))

bottom = [0] * len(top6)
for metric, color in zip(metrics, stack_colors):
    ax.bar(top6["team"], top6[metric], bottom=bottom, label=metric.capitalize(), color=color)
    bottom = [b + v for b, v in zip(bottom, top6[metric])]

ax.set_xlabel("Club")
ax.set_ylabel("Matches")
ax.set_title("Top 6 Clubs — Wins / Draws / Losses (2023/24)")
ax.legend()

plt.tight_layout()
plt.savefig("plot_top6_stacked.png", dpi=150)
plt.show()

---
## Part 5 — Export

In [ ]:
# Export the full cleaned dataset
df.to_csv("pl_2023_24_standings_clean.csv", index=False)
print("Cleaned data exported to pl_2023_24_standings_clean.csv")

In [ ]:
# Export a separate summary table for the top 6
top6_summary = top6[["position", "team", "points", "goals_scored",
                      "goals_conceded", "goal_diff", "win_rate"]]
top6_summary.to_csv("pl_2023_24_top6.csv", index=False)
print("Top 6 summary exported to pl_2023_24_top6.csv")
print()
try:
    display(top6_summary)
except NameError:
    print(top6_summary)

---
## Part 6 — Conclusion

This project analysed the Premier League 2023/24 final standings using live data from the API-Football API.

**Key findings:**

1. **Goals scored strongly predicts final points.** The Pearson correlation between goals scored and points was high, confirming that attacking output is the biggest separator between clubs.

2. **Defensive record is equally telling.** Relegated clubs all sat above the diagonal in the scatter plot, meaning they conceded far more than they scored.

3. **A clear two-tier structure exists.** The top 4 clubs had significantly higher win rates and goal differences compared to mid-table teams.

4. **Differences within the top 6 are subtle.** Even at the top, the number of draws and losses distinguishes Champions League qualifiers from Europa League finishers.

**Limitations:** This analysis uses only final standings data. Match-by-match or per-player statistics would allow deeper analysis, such as home vs away performance or form over the season.

---

## Statement of AI Use

Claude (Anthropic) was used to help structure this notebook, suggest appropriate chart types, and debug code. All analysis decisions, interpretations, and conclusions were written independently. The data is real and sourced directly from the API-Football API.